In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

bronze = "healthcare_adjudication.healthcare_adjudication_bronze"
silver = "healthcare_adjudication.healthcare_adjudication_silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver}")


In [0]:
providers = (
    spark.table(f"{bronze}.providers")
    .dropDuplicates(["provider_id"])
)

providers.write.format("delta").mode("overwrite") \
.saveAsTable(f"{silver}.providers")


In [0]:
payers = (
    spark.table(f"{bronze}.payers")
    .dropDuplicates(["payer_id"])
)

payers.write.format("delta").mode("overwrite") \
.saveAsTable(f"{silver}.payers")


In [0]:
patients = (
    spark.table(f"{bronze}.patients")
    .withColumn("dob", to_date("dob"))
    .dropDuplicates(["patient_id"])
)

patients.write.format("delta").mode("overwrite") \
.saveAsTable(f"{silver}.patients")


In [0]:
policies = (
    spark.table(f"{bronze}.policies")
    .select(
        col("policy_id"),
        col("member.patient_id").alias("patient_id"),
        col("payer.payer_id").alias("payer_id"),
        col("coverage.plan").alias("plan"),
        col("coverage.limit").alias("coverage_limit"),
        col("coverage.copay").alias("copay")
    )
)

policies.write.format("delta").mode("overwrite") \
.saveAsTable(f"{silver}.policies")

In [0]:
%sql
--use catalog healthcare_adjudication
use schema healthcare_adjudication_silver

In [0]:
%sql
select * from policies limit 10

Claims

In [0]:
claims = (
    spark.table(f"{bronze}.claims")
    .withColumn("claim_date", to_date("claim_date"))
    .dropDuplicates(["claim_id"])
)

claims.write.format("delta").mode("overwrite") \
.saveAsTable(f"{silver}.claims")


In [0]:

claim_lines = (
    spark.table(f"{bronze}.claim_lines")
    .withColumn("procedure", explode("procedures"))
    .select(
        col("claim_id"),
        col("procedure.code").alias("procedure_code"),
        col("procedure.charge").alias("procedure_charge")
    )
)

claim_lines.write.format("delta").mode("overwrite") \
.saveAsTable(f"{silver}.claim_lines")



In [0]:
payments = (
    spark.table(f"{bronze}.payments")
    .dropDuplicates(["payment_id"])
)

payments.write.format("delta").mode("overwrite") \
.saveAsTable(f"{silver}.payments")


In [0]:
%sql
use healthcare_adjudication.healthcare_adjudication_gold

In [0]:
%sql
select * from claim_approval_rate limit10

In [0]:
%sql
use healthcare_adjudication_silver

In [0]:
%sql
select * from providers where provider_id = 2001
